# 00. Registro de fuentes y contrato de reproducibilidad

Registramos los insumos disponibles antes de transformarlos y delimitamos la
versión del ETL que los procesa. El contrato incluye las huellas SHA-256 de
las fuentes, las políticas de limpieza, las versiones del entorno y el código
productor. Esta identificación permite distinguir cambios de datos de cambios
de procedimiento y conservar la evidencia de cada generación.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq
from IPython.display import display, Markdown

BASE = Path.cwd().resolve()
PROJECT = next(p / "4_proyecto" for p in [BASE, *BASE.parents] if (p / "4_proyecto/V3/data").is_dir())
V3 = PROJECT / "V3"
sys.path.insert(0, str(PROJECT))
from V3.src.etl_contract import build_manifest, verify_manifest, provenance, get_processed_root, get_quarantine_root
from V3.src.enterprise_etl import run_etl, verify_outputs
print("Directorio del proyecto:", V3)

Directorio del proyecto: /Users/oscarmartinez/Desktop/Caso_De_negocio_ITAM/V3


## Inventario del contenido de RAW

Calculamos las huellas a partir del contenido de los archivos y mantenemos
RAW en modo de lectura. Excluimos únicamente `.DS_Store`, que contiene
metadatos de Finder y no observaciones. Las copias extraídas de DENUE y GTFS
se registran en el inventario y se verifican contra el miembro correspondiente
de su ZIP; la lectura analítica se realiza una sola vez desde el archivo
comprimido. Una diferencia entre copia y archivo original detiene el ETL.

In [2]:
manifest = build_manifest(V3)
sources = pd.DataFrame(manifest["files"])
sources["bytes"] = sources.bytes.astype("Int64")
sources = provenance(sources, sources.path.astype("string"), manifest)
display(sources)
display(Markdown(f"**Resultados.** Registramos {len(sources):,} archivos de datos y documentación. "
                 f"El contrato activo es `{manifest['contract_id']}`."))

,bytes,path,role,sha256,_source_file,_processed_at
0,226315,data/raw/ecobici_stations/station_information....,core_etl,8b4eaa98653beea3d51dc3d894ea8d5ee749923daee40d...,data/raw/ecobici_stations/station_information....,2026-10-11 23:49:59+00:00
1,110988817,data/raw/ecobici_trips/2025-01.csv,core_etl,6b23c3d000856dabb72802eeb6cc84fdbbe0e5d1046019...,data/raw/ecobici_trips/2025-01.csv,2026-10-11 23:49:59+00:00
2,107105837,data/raw/ecobici_trips/2025-02.csv,core_etl,f61d8be8138e1f6f922335b807626db6649d7936bc859a...,data/raw/ecobici_trips/2025-02.csv,2026-10-11 23:49:59+00:00
3,112443905,data/raw/ecobici_trips/2025-03.csv,core_etl,a7cd8385505b596f895976bfb42d78dc4204c1b6202559...,data/raw/ecobici_trips/2025-03.csv,2026-10-11 23:49:59+00:00
4,106058647,data/raw/ecobici_trips/2025-04.csv,core_etl,d4cb45a4280c486ce905bfa932b9562731356cb979112e...,data/raw/ecobici_trips/2025-04.csv,2026-10-11 23:49:59+00:00
5,109082864,data/raw/ecobici_trips/2025-05.csv,core_etl,943aa9677b9440abd1a83fce52946c5909c6699f82e287...,data/raw/ecobici_trips/2025-05.csv,2026-10-11 23:49:59+00:00
6,96821733,data/raw/ecobici_trips/2025-06.csv,core_etl,e1f6f5f07436aa03668811f1f7e0d00aeeff411eeafbb6...,data/raw/ecobici_trips/2025-06.csv,2026-10-11 23:49:59+00:00
7,102053921,data/raw/ecobici_trips/2025-07.csv,core_etl,0a8bf13ce0b0bf1ba395435a07178bf7ac4381773bd63b...,data/raw/ecobici_trips/2025-07.csv,2026-10-11 23:49:59+00:00
8,98785437,data/raw/ecobici_trips/2025-08.csv,core_etl,9be378627a1b4244a98d357c4d9c7a74570566b5598449...,data/raw/ecobici_trips/2025-08.csv,2026-10-11 23:49:59+00:00
9,97279076,data/raw/ecobici_trips/2025-09.csv,core_etl,ea5bafb251c93265a759fc2ce3d43b136d4fa75938af52...,data/raw/ecobici_trips/2025-09.csv,2026-10-11 23:49:59+00:00


**Resultados.** Registramos 37 archivos de datos y documentación. El contrato activo es `6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd5ca89de9cdb26afd93`.

## Políticas y alcance de la versión

La versión actual normaliza estaciones, AGEBs, viajes, los tabulados del Censo,
DENUE, GTFS, ciclovías y registros viales C5, de accidentes y de víctimas SSC.
Los diccionarios, metadatos y publicaciones se conservan como documentación
estructurada, no como observaciones. No completamos nulos por defecto:
distinguimos faltantes legítimos, valores suprimidos, atributos opcionales y
ausencia de campos necesarios para una tarea.

El año 2025 delimita las observaciones de viajes e incidentes. Los catálogos
y las fuentes estructurales conservan su época de origen: Censo 2020, marco
geográfico diciembre de 2025, catálogo GBFS septiembre de 2026 y DENUE 2026.
El calendario GTFS contiene servicios de diferentes años. Estas diferencias
se mantienen como limitaciones temporales y no se reinterpretan como mediciones
simultáneas de 2025.

In [3]:
display(pd.Series(manifest["policy"], name="politica").to_frame())
display(pd.Series(manifest["versions"], name="version").to_frame())
display(pd.Series(manifest["code_hashes"], name="sha256_productor").to_frame())
display(sources.groupby("role").size().rename("archivos").to_frame())

,politica
activity_cohort,each valid 2025 endpoint independently; compou...
ageb_default_declared_crs,EPSG:6365
auxiliary_sources,all registered datasets normalized; documents ...
calendar_days,365
census,all geographic levels retained; AGEB totals ar...
cycling,layers stay separate; total layer not concaten...
daily_activity,(withdrawal_weight + arrival_weight) / (2 * 365)
dedup,latest_valid_update_then_sha256_then_source_fi...
denue,snapshot 2026 retained and flagged; registrati...
gtfs,optional nulls and times beyond24h allowed; fo...


,version
beautifulsoup4,4.14.3
geopandas,1.1.2
numpy,1.26.4
pandas,2.3.3
pyarrow,23.0.1
pyogrio,0.11.1
pyproj,3.7.1
shapely,2.1.2


,sha256_productor
artifact_reader.py,a7a2508223d8ab34738ea766ef70a3616dfcb11268f2c6...
auxiliary_etl.py,6ee788a7aba07345f7206ade9f0854be7cbbc711e51b31...
enterprise_etl.py,da66b87a015e29dff59260186d220274f97bad3ca7903b...
etl_contract.py,e102f3e451b61ccd641089ab2f8407f91df57ac6f8cd52...


,archivos
role,
auxiliary_etl_or_document,11
core_etl,15
verified_mirror,11


## Verificación del contrato

Repetimos la comprobación de RAW para asegurar que el contrato permanece
vigente. Las generaciones anteriores no se eliminan ni se reinterpretan:
mantienen sus auditorías y artefactos. Una modificación requiere una
actualización explícita de versión. El timestamp UTC de las tablas pertenece
a la época lógica del contrato; las horas reales de ejecución se registran
separadamente, de modo que no alteran los archivos de datos en cada corrida.

In [4]:
assert verify_manifest(V3) == manifest
display(Markdown("**Resultados.** El contrato conserva las mismas fuentes, políticas "
                 "y versiones. La verificación de contenido no detectó cambios en RAW."))

**Resultados.** El contrato conserva las mismas fuentes, políticas y versiones. La verificación de contenido no detectó cambios en RAW.